In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dill
import torch



In [ ]:
from sentence_transformers import CrossEncoder,InputExample

from torch.utils.data import DataLoader

from src.metric import model_evaluation


In [ ]:
with open('../data/cleaned/clean_train_df.dill','rb') as f:
    train_df=dill.load(f)
    
with open('../data/cleaned/clean_val_df.dill','rb') as f:
    val_df=dill.load(f)
        
with open('../data/cleaned/clean_test_df.dill','rb') as f:
    test_df=dill.load(f)
    


In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
label_to_score = {0: 0.0, 1: 0.6, 2: 1.0} #1 map to 0.6 bec in resume ranking maybe is more closs to confirm close than no job acc to jd


cross_train_examples=[
    InputExample(texts=[j,r],label=float(label_to_score[l]))
    for r,j,l in zip(train_df['resume_text'],train_df['job_description_text'],train_df['label'])
]

print(f"total training example:{len(cross_train_examples)}")

In [ ]:
cross_train_dataloader=DataLoader(cross_train_examples,shuffle=True,batch_size=32)

In [ ]:
cross_encoder_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2',num_labels=1,device=device, max_length=384)

In [ ]:
epochs=4
best_score=float('-inf')
print("\tTraining Phase")
for epoch in range(1,epochs+1):
    print(f"Epoch: {epoch}----------")

    cross_encoder_model.fit(train_dataloader=cross_train_dataloader,epochs=1,
                             warmup_steps=int(len(cross_train_dataloader) * epochs * 0.1),
                            show_progress_bar=True)
    
    val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

    scores=cross_encoder_model.predict(val_pairs,batch_size=32,show_progress_bar=False)
    
    metrics=model_evaluation(scores,val_df,'job_description_text')

    print(f"Spearman Score:{metrics['spearman_score']}")
    print(f"Top 3 score:{metrics['topk_score']}")
    print(f"NDGC Score:{metrics['ndcg_val']}")
    print(f"MAP Score:{metrics['mrr_score']}")
    print(f"MRR Score:{metrics['map_score']}")

    final_score=0.6*metrics['ndgc']+0.3*metrics['map']+0.1*(metrics['mrr']+metrics['topk'])

    if final_score>best_score:
        best_score=final_score
        cross_encoder_model.save("cross_encoder_model")
    


In [ ]:
print("\tInference Phase")

val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

scores=cross_encoder_model.predict(val_pairs,batch_size=32,show_progress_bar=False)

model_evaluation(scores,val_df,'job_description_text')

In [ ]:
ranked_result=[]
val_df_new=val_df.copy()
val_df_new['score']=scores

for jd,group in val_df_new.groupby('job_description_text'):
    ranked_group=group.sort_values("score",ascending=False)
    ranked_result.append(ranked_group)

final_rank_df=pd.concat(ranked_result)

In [ ]:
i=0
for jd,group in final_rank_df.groupby('job_description_text'):
    if(len(group)>2 and len(group)<10):
        print("Job Description:\n",jd[:300])
        print(group[['label','score']])
        i+=1
        if i==3:
            break

In [ ]:
test_pairs=list(zip(test_df['job_description_text'],test_df['resume_text']))

scores=cross_encoder_model.predict(test_pairs,batch_size=32,show_progress_bar=False)

model_evaluation(scores,test_df,'job_description_text')